In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-text-splitters langchain-experimental openai faiss-cpu pypdf python-dotenv pandas tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing. Update the .env file before running the examples.")
print("API key loaded and ready to use.")

API key loaded and ready to use.


## RAG over *Os Elementos*

1. Load and chunk the PDF with LangChain helpers.
2. Embed the chunks with OpenAI embeddings and store them in a FAISS index.
3. Use a chat model to answer questions via a retrieval QA chain.

In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

pdf_path = Path("OsElementos-Euclides.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found at {pdf_path}.")

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()
print(f"Loaded {len(documents)} pages from the PDF.")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} text chunks.")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Você é um assistente que responde em português usando exclusivamente o contexto fornecido."),
        ("human", "Contexto:\n{context}\n\nPergunta:\n{question}"),
    ]
)

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever
    }
    | prompt
    | llm
    | StrOutputParser()
 )

question = "Resuma os principais conceitos geométricos apresentados nas primeiras seções de Os Elementos."
response = rag_chain.invoke(question)
print(response)

Loaded 595 pages from the PDF.
Created 1568 text chunks.
Nos primeiros capítulos de "Os Elementos" de Euclides, são apresentados conceitos fundamentais da geometria, que incluem:

1. **Definições**: São descrições precisas de termos geométricos, como pontos, linhas, superfícies e ângulos, que servem como base para o desenvolvimento da geometria.

2. **Postulados**: São proposições aceitas como verdadeiras sem necessidade de demonstração, que servem como fundamentos para a construção de teoremas. Um exemplo clássico é o postulado que afirma que é possível traçar uma linha reta entre dois pontos.

3. **Axiomas (ou noções comuns)**: São princípios gerais que se aplicam a todas as áreas da matemática, como a ideia de que coisas iguais a um mesmo são iguais entre si.

4. **Teoremas**: São proposições que podem ser demonstradas a partir dos postulados e definições. A estrutura lógica da geometria é construída a partir da dedução de teoremas a partir desses princípios.

Esses conceitos formam

In [6]:
follow_up = "Quais definições ou postulados centrais sustentam o desenvolvimento inicial do texto?"
follow_up_answer = rag_chain.invoke(follow_up)
print(follow_up_answer)

O desenvolvimento inicial do texto é sustentado por definições e postulados centrais que incluem:

1. **Recensio**: A fase de pesquisa e coleta de todo o material existente de uma obra, que forma sua tradição, seja direta (manuscritos) ou indireta (fontes, traduções, citações, etc.).

2. **Estemática**: A fase que busca descobrir a origem e a ascendência dos textos, referida por Lachmann como "originem detegere".

3. **Emendatio**: A fase de emendar e corrigir o texto, visando trazer à luz o original ou o texto autógrafo.

Essas fases são fundamentais para a crítica textual e a fixação do texto, conforme discutido no contexto apresentado.


## Question answering over CSV data

The next section shows how to let an OpenAI model reason over the `microdados_servidores_2023.csv` file with the help of a pandas dataframe agent.

In [7]:
import pandas as pd

csv_path = Path("microdados_servidores_2023.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found at {csv_path}.")

df = pd.read_csv(csv_path, sep=";", encoding="utf-8")
df["Número de registros"] = pd.to_numeric(df["Número de registros"], errors="coerce").fillna(0)
print(df.shape)
df.head()

(83496, 15)


,Cod Unidade,Código da Unidade de Ensino - SISTEC,Classe,Código Municipio com DV,Instituição,Jornada de Trabalho,Município,Número de registros,Região,RSC,Titulação,Unidade de Lotação,Vinculo Carreira,Vinculo Contrato,Vinculo Professor
0,248,31396.0,D,3304557,CPII,DE,Rio de Janeiro,1,NaN,Não Possui,Doutorado,Campus Humaitá II,Ebtt,Efetivo,Sim
1,212,44132.0,D,3304557,CPII,DE,Rio de Janeiro,1,NaN,Não Possui,Doutorado,Campus Engenho Novo I,Ebtt,Efetivo,Sim
2,594,NaN,E,3304557,CPII,40h,Rio de Janeiro,1,NaN,Não Possui,Mestrado,Reitoria do Colégio Pedro II,Pcctae,Efetivo,Não
3,171,1699.0,D,3304557,CPII,40h,Rio de Janeiro,1,NaN,Não Possui,Especialização,Campus Centro,Pcctae,Efetivo,Não
4,171,1699.0,D,3304557,CPII,40h,Rio de Janeiro,1,NaN,Não Possui,Especialização,Campus Centro,Pcctae,Efetivo,Não


In [9]:
from langchain_experimental.agents import create_pandas_dataframe_agent

pandas_agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True,  # opt into the Python REPL tool that runs arbitrary code
 )

ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.

In [ ]:
question_regioes = (
    "Calcule o total de servidores (soma da coluna 'Número de registros') por Região e apresente em ordem decrescente."
)
resposta_regioes = pandas_agent.invoke({"input": question_regioes})
print(resposta_regioes["output"])

In [ ]:
question_municipio = (
    "Quais são os três municípios com mais registros e quais Instituições estão associados a eles?"
)
resposta_municipio = pandas_agent.invoke({"input": question_municipio})
print(resposta_municipio["output"])